# 01 — Experimentos ML (features wavelet → classificadores)

Extrai features wavelet **multivariadas** (concatenadas por canal), faz busca de
hiperparâmetros com `RandomizedSearchCV` (scoring `f1_macro`) e avalia no teste.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
sys.path.insert(0, 'config')
from experiment_config import (
    UWAVE_CONFIG, DATA_DIR, RESULTS_DIR, SEED,
    WAVELET_CONFIG, LEARNED_WAVELET_CONFIG, DL_TRAINING_CONFIG,
    ML_MODELS_CONFIG, ML_SEARCH_CONFIG, build_param_dist, N_JOBS_QUARTER,
)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(SEED)
print('Dataset:', UWAVE_CONFIG['dataset_name'], '| classes:', UWAVE_CONFIG['n_classes'],
      '| seq_len:', UWAVE_CONFIG['sequence_length'], '| canais:', UWAVE_CONFIG['n_features'])

## 1. Carregar dados e features

In [ ]:
from src.data_loader import UWaveDataLoader
from src.feature_extraction import WaveletFeatureExtractor
d = UWaveDataLoader.load_prepared(DATA_DIR)
X_train, y_train, X_test, y_test = d['X_train'], d['y_train'], d['X_test'], d['y_test']
extractor = WaveletFeatureExtractor(wavelet=WAVELET_CONFIG['wavelet_type'],
                                    level=WAVELET_CONFIG['decomposition_level'])
Xtr = extractor.extract_features(X_train)
Xte = extractor.extract_features(X_test)
print('features:', Xtr.shape, Xte.shape)

## 2. Normalização

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

## 3. Busca de hiperparâmetros + avaliação (multiclasse)

> `n_jobs` da busca limitado a `N_JOBS_QUARTER`: cada estimador já paraleliza
> internamente e `src.models` importa TensorFlow — evita oversubscription/OOM.

In [ ]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from src.models import (create_rf_classifier, create_xgb_classifier, create_lgbm_classifier)
from src.evaluation import ClassificationEvaluator

FACTORIES = {'RandomForest': create_rf_classifier,
             'XGBoost': create_xgb_classifier,
             'LightGBM': create_lgbm_classifier}
ev = ClassificationEvaluator()
cv = StratifiedKFold(n_splits=ML_SEARCH_CONFIG['cv_splits'], shuffle=True, random_state=SEED)
search_jobs = max(1, min(N_JOBS_QUARTER, 4))
rows = {}
for name, factory in FACTORIES.items():
    spec = ML_MODELS_CONFIG[name]
    search = RandomizedSearchCV(factory(), build_param_dist(spec['param_dist']),
        n_iter=spec['n_iter'], scoring=ML_SEARCH_CONFIG['scoring'], cv=cv,
        random_state=SEED, n_jobs=search_jobs)
    search.fit(Xtr_s, y_train)
    best = search.best_estimator_
    proba = best.predict_proba(Xte_s)
    pred = np.argmax(proba, axis=1)
    rows[name] = ev.evaluate(y_test, pred, y_proba=proba, prefix='test')
    print(f'{name:14s} acc={rows[name]["test_accuracy"]:.4f}  f1_macro={rows[name]["test_f1_macro"]:.4f}  auc_ovr={rows[name]["test_auc_ovr"]:.4f}')
pd.DataFrame(rows).T.round(4)

> Para os modelos DL (raw / db4 / learned wavelet) use os notebooks 02–04 ou,
> de preferência, a **fila GPU**: `python run_dl_queue.py --fresh`.